In [1]:
from sqlalchemy import create_engine
import pandas as pd

# 数据库连接信息
user = "root"
password = "20041025"  # 替换为你的密码
host = "localhost"
port = "3306"
database = "ucb_db"

# 创建 SQLAlchemy 引擎
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# 假设你想读取一个叫 books 的表
df_books = pd.read_sql("SELECT * FROM books_info;", engine)
df_books.head()


,title,subtitle,author,rating_number,features,description,price,store,categories,details,book_id
0,Chaucer,"Hardcover – Import, January 1, 2004","{""avatar"": ""https://m.media-amazon.com/images/...",29,[],[],8.23,Peter Ackroyd (Author),"[""Books"", ""Literature & Fiction"", ""History & C...","Published by Chatto & Windus, the first editio...",bookid_1
1,Notes from a Kidwatcher,First Edition,"{""avatar"": ""https://m.media-amazon.com/images/...",1,"[""Contains 23 selected articles by this influe...","[""About the Author"", ""SANDRA WILDE, Ph.D., is ...",3.52,Sandra Wilde (Editor),"[""Books"", ""Reference"", ""Words, Language & Gram...","This book, published by Heinemann in its first...",bookid_2
2,Service: A Navy SEAL at War,"Hardcover – May 8, 2012","{""avatar"": ""https://m.media-amazon.com/images/...",3421,"[""Marcus Luttrell, author of the #1 bestseller...","[""Review"", ""Praise for SERVICE\""An action-pack...",17.17,"Marcus Luttrell (Author), James D. Hornfischer","[""Books"", ""Biographies & Memoirs"", ""Leaders & ...","This book, published by Little, Brown and Comp...",bookid_3
3,Monstrous Stories #4: The Day the Mice Stood S...,"Paperback – October 29, 2013",None,40,"[""Funny, light-hearted monster stories that ar...",[],7.43,Dr. Roach (Author),"[""Books"", ""Children's Books"", ""Science Fiction...","This book, published by Scholastic Paperbacks ...",bookid_4
4,Parker & Knight,Kindle Edition,"{""avatar"": ""https://m.media-amazon.com/images/...",381,"[""From REMINGTON KANE, the author of The Taken...",[],0.00,Remington Kane (Author) Format: Kindle Edition,"[""Books"", ""Mystery, Thriller & Suspense"", ""Thr...","The book was published on May 18, 2014, and is...",bookid_5


In [20]:
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
from openai import AzureOpenAI
# MySQL配置
mysql_password = "20041025"  # 替换为你的MySQL密码
mysql_db = "ucb_db"
mysql_table = "books_info"

# ========== Configuration ========== #
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o-mini" 

def fetch_business_data():
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password=mysql_password,
            database=mysql_db
        )
        cursor = conn.cursor()
        cursor.execute(f"SELECT * FROM {mysql_table}")
        column_names = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()
        conn.close()
        return rows, column_names
    except Exception as e:
        print(f"❌ MySQL connection failed: {e}")
        return [], []

rows, column_names = fetch_business_data()
books = [dict(zip(column_names, row)) for row in rows]
print(f"✅ Total {len(books)} books")


def get_decade_by_llm(details, client, deployment_name):
    prompt = (
        "Given the following book details, extract the decade in which the book was first published.\n"
        "Respond with a single answer in the format like '1980s', '2000s', etc. If uncertain because no year information is included in the book details, just answer Uncertain.\n\n"
        f"Details: {details}"
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts publishing decades from book descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return "Unknown"

for i, book in enumerate(books):
    details = book.get("details", "")
    if not details or details.strip() == "":
        book["decade"] = "Unknown"
        print(f"❌ [{i}] Unknown")
        continue
    decade = get_decade_by_llm(details, client, deployment_name)
    book["decade"] = decade
    print(f"✅ [{i}] {book.get('title', 'Untitled')} → {decade}")
book_id_to_decade = {b["book_id"]: b["decade"] for b in books}
# 构造 DataFrame（方便后续分析）
df_books = pd.DataFrame(books)
print(df_books["decade"].value_counts())  # 看每个 decade 的分布


✅ Total 200 books
✅ [0] Chaucer → 2000s
✅ [1] Notes from a Kidwatcher → 1990s
✅ [2] Service: A Navy SEAL at War → 2010s
✅ [3] Monstrous Stories #4: The Day the Mice Stood Still → Uncertain
✅ [4] Parker & Knight → 2010s
✅ [5] Writings from a Black Woman Living in the Land of the "Free": Strength, Power, Resilience → 2020s
✅ [6] Child Development: A Practitioner's Guide:2nd (Second) edition → 1990s
✅ [7] Make: Electronics: Learning Through Discovery → 2010s
✅ [8] Reunion: The Children of Lauderdale Park → 2010s
✅ [9] Four Centuries of American Education → 2000s
✅ [10] Mining Engineers and the American West: The Lace-Boot Brigarde, 1849-1933 → 1990s
✅ [11] Heart of Silk and Shadows: A Fae Fantasy Romance (Fae Isles) → 2020s
✅ [12] Girl Made of Glass → 2020s
✅ [13] The Old Man and the Pirate Princess → 2010s
✅ [14] Japanese Girls and Women → 2000s
✅ [15] Behavior Principles in Everyday Life → 1990s
✅ [16] PQL 3 - Lola (Spanish Edition) → 1980s
✅ [17] A sermon, preached at the execution of 

In [21]:
import sqlite3

def fetch_reviews_from_sqlite(sqlite_path):
    try:
        conn = sqlite3.connect(sqlite_path)
        query = "SELECT * FROM review"
        df_review = pd.read_sql(query, conn)
        conn.close()
        return df_review
    except Exception as e:
        print(f"❌ SQLite load failed: {e}")
        return pd.DataFrame()

# 路径按你实际位置调整
df_review = fetch_reviews_from_sqlite("../query_dataset/review_query.db")
print(f"✅ Loaded {len(df_review)} reviews")


✅ Loaded 1833 reviews


In [22]:

import json

def get_mapping_rule_from_llm(book_ids, purchase_ids, client, deployment_name):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `purchase_id`: {json.dumps(purchase_ids, indent=2)}\n"
        f"- The second column is `book_id`: {json.dumps(book_ids, indent=2)}\n\n"
        "Each purchase_id corresponds to a book_id, but the mapping rule is not provided.\n"
        "Your task is to determine the mapping relationship between the two sets of IDs.\n"
        "For example, is the number in `purchaseid_123` always equal to the number in `bookid_23` plus an offset?\n\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English \n"
        "2. An explanation of why you think this rule holds."
    )

    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two columns of IDs for structural relationships."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )

    return response.choices[0].message.content.strip()


book_ids = [b["book_id"] for b in books]
purchase_ids = df_review["purchase_id"].drop_duplicates().tolist()

mapping_rule_explanation = get_mapping_rule_from_llm(book_ids, purchase_ids, client, deployment_name)
print("🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)


🧠 GPT-inferred mapping rule:

1. **Mapping Rule**: The mapping relationship between `purchase_id` and `book_id` is that the number in `purchaseid_X` corresponds to the number in `bookid_Y` where Y is equal to X. In other words, the mapping is direct and one-to-one, meaning `purchaseid_n` maps to `bookid_n` for all n from 1 to 200.

2. **Explanation**: 
   - To verify this mapping rule, we can observe the structure of both ID columns. Each `purchase_id` has the format `purchaseid_X` and each `book_id` has the format `bookid_Y`, where X and Y are integers ranging from 1 to 200.
   - By examining the IDs, we can see that the numbers in both columns are sequential and cover the same range (1 to 200). This suggests a direct correspondence.
   - For example, `purchaseid_1` corresponds to `bookid_1`, `purchaseid_2` corresponds to `bookid_2`, and so on, up to `purchaseid_200` corresponding to `bookid_200`.
   - There are no gaps or offsets in the numbering, which further supports the conclusio

In [23]:
def resolve_purchase_to_book_via_llm(purchase_id, mapping_rule_text, client, deployment_name):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_text}\n\n"
        f"Now, based on this rule, please determine the corresponding `book_id` for the following `purchase_id`:\n"
        f"- purchase_id: {purchase_id}\n\n"
        "Respond only with the `book_id`, e.g., 'bookid_88'."
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps purchase IDs to book IDs using a known rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {purchase_id}: {e}")
        return None

predicted_book_ids = []

for i, row in df_review.iterrows():
    purchase_id = row["purchase_id"]
    predicted_book_id = resolve_purchase_to_book_via_llm(purchase_id, mapping_rule_explanation, client, deployment_name)
    
    predicted_book_ids.append(predicted_book_id)
    print(f"[{i}] purchase_id = {purchase_id} → book_id = {predicted_book_id}")
    
df_review["book_id"] = predicted_book_ids


[0] purchase_id = purchaseid_186 → book_id = bookid_186
[1] purchase_id = purchaseid_191 → book_id = bookid_191
[2] purchase_id = purchaseid_190 → book_id = bookid_190
[3] purchase_id = purchaseid_8 → book_id = bookid_8
[4] purchase_id = purchaseid_178 → book_id = bookid_178
[5] purchase_id = purchaseid_186 → book_id = bookid_186
[6] purchase_id = purchaseid_76 → book_id = bookid_76
[7] purchase_id = purchaseid_186 → book_id = bookid_186
[8] purchase_id = purchaseid_115 → book_id = bookid_115
[9] purchase_id = purchaseid_167 → book_id = bookid_167
[10] purchase_id = purchaseid_188 → book_id = bookid_188
[11] purchase_id = purchaseid_23 → book_id = bookid_23
[12] purchase_id = purchaseid_196 → book_id = bookid_196
[13] purchase_id = purchaseid_196 → book_id = bookid_196
[14] purchase_id = purchaseid_3 → book_id = bookid_3
[15] purchase_id = purchaseid_48 → book_id = bookid_48
[16] purchase_id = purchaseid_154 → book_id = bookid_154
[17] purchase_id = purchaseid_99 → book_id = bookid_99


In [24]:
import pandas as pd

# === Step 1: 将每条 review 映射到对应 decade ===
df_review["decade"] = df_review["book_id"].map(book_id_to_decade)

# === Step 2: 清洗无效值 ===
df_review["rating"] = pd.to_numeric(df_review["rating"], errors="coerce")
df_review = df_review[df_review["rating"].notnull()]
df_review = df_review[df_review["decade"].notnull() & (df_review["decade"] != "Unknown")]

# === Step 3: 按 decade 聚合，统计每个 decade 的平均评分和不同 book 数量 ===
df_result = (
    df_review.groupby("decade")
    .agg(
        num_books=("book_id", "nunique"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

# === Step 4: 只保留包含 ≥10 本书的 decade ===
df_filtered = df_result[df_result["num_books"] >= 10]

# === Step 5: 找出平均评分最高的 decade ===
top_decade = df_filtered.sort_values(by="avg_rating", ascending=False).head(1)

# === Step 6: 输出结果 ===
print("📊 Aggregated results by decade (≥10 distinct books):")
display(df_filtered.sort_values(by="avg_rating", ascending=False))

print("\n🏆 Top-rated decade:")
display(top_decade)


📊 Aggregated results by decade (≥10 distinct books):


,decade,num_books,avg_rating
8,2020s,21,4.663636
9,Uncertain,14,4.611429
7,2010s,82,4.602170
6,2000s,46,4.273684
4,1980s,11,4.208333
5,1990s,16,3.779661



🏆 Top-rated decade:


,decade,num_books,avg_rating
8,2020s,21,4.663636


In [25]:
import mysql.connector
import pandas as pd
from sqlalchemy import create_engine
from openai import AzureOpenAI
# MySQL配置
mysql_password = "20041025"  # 替换为你的MySQL密码
mysql_db = "ucb_db"
mysql_table = "books_info"

# ========== Configuration ========== #
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o-mini" 

def fetch_business_data():
    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password=mysql_password,
            database=mysql_db
        )
        cursor = conn.cursor()
        cursor.execute(f"SELECT * FROM {mysql_table}")
        column_names = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()
        conn.close()
        return rows, column_names
    except Exception as e:
        print(f"❌ MySQL connection failed: {e}")
        return [], []

rows, column_names = fetch_business_data()
books = [dict(zip(column_names, row)) for row in rows]
print(f"✅ Total {len(books)} books")

def is_qualified_book(details, categories, client, deployment_name):
    prompt = (
        "You're given a book's description and category field.\n"
        "Determine if this book is written in English and belongs to the 'Literature & Fiction' category.\n\n"
        "If both conditions are true, answer with 'yes'. If either is false, answer with 'no'.\n\n"
        f"Description:\n{details}\n\n"
        f"Categories:\n{categories}\n\n"
        "Just answer 'yes' or 'no'.\n\n"
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that filters books by language and category."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower() == "yes"
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return False

qualified_books = []

for i, book in enumerate(books):
    details = book.get("details", "")
    categories = book.get("categories", "")
    if not details.strip():
        print(f"❌ [{i}] No details")
        continue

    is_qualified = is_qualified_book(details, categories, client, deployment_name)
    if is_qualified:
        qualified_books.append(book)
        print(f"✅ [{i}] Matched: {book.get('title', 'Untitled')}, {categories}, {details}")
    else:
        print(f"❌ [{i}] Not matched")

df_qualified = pd.DataFrame(qualified_books)
print(f"\n Retained {len(df_qualified)} English Literature & Fiction books")


✅ Total 200 books
✅ [0] Matched: Chaucer, ["Books", "Literature & Fiction", "History & Criticism"], Published by Chatto & Windus, the first edition of this book was released on January 1, 2004. It is written in English and comes in a hardcover format, comprising 196 pages. The book has an ISBN-10 of 0701169850 and an ISBN-13 of 978-0701169855. Weighing 10.1 ounces, its dimensions are 5.39 x 0.71 x 7.48 inches.
❌ [1] Not matched
❌ [2] Not matched
❌ [3] Not matched
❌ [4] Not matched
❌ [5] Not matched
❌ [6] Not matched
❌ [7] Not matched
✅ [8] Matched: Reunion: The Children of Lauderdale Park, ["Books", "Literature & Fiction", "Genre Fiction"], This book, published independently on September 25, 2019, is written in English and spans 367 pages. It is available in paperback format and has an ISBN-10 of 1694621731 and an ISBN-13 of 978-1694621733. The item weighs 1.38 pounds and measures 6 x 0.92 x 9 inches.
❌ [9] Not matched
❌ [10] Not matched
❌ [11] Not matched
✅ [12] Matched: Girl Made of 

In [27]:
import sqlite3
import json

# ========== Step 3: Load reviews from SQLite ========== #
def fetch_reviews():
    try:
        conn = sqlite3.connect("../query_dataset/review_query.db")
        query = "SELECT * FROM review"
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ Loaded {len(df)} reviews")
        return df
    except Exception as e:
        print(f"❌ SQLite load failed: {e}")
        return pd.DataFrame()

df_review = fetch_reviews()

# ========== Step 4: Infer purchase_id → book_id mapping rule ========== #
def get_mapping_rule(book_ids, purchase_ids):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `purchase_id`: {json.dumps(purchase_ids, indent=2)}\n"
        f"- The second column is `book_id`: {json.dumps(book_ids, indent=2)}\n\n"
        "Each purchase_id corresponds to a book_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English \n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two columns of IDs for structural relationships."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

book_ids = [b["book_id"] for b in books]
purchase_ids = df_review["purchase_id"].drop_duplicates().tolist()
mapping_rule_explanation = get_mapping_rule(book_ids, purchase_ids)
print("\n🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)

# ========== Step 5: Apply rule to each review ========== #
def resolve_purchase_to_book(purchase_id):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now, based on this rule, please determine the corresponding `book_id` for the following `purchase_id`:\n"
        f"- purchase_id: {purchase_id}\n\n"
        "Respond only with the `book_id`, e.g., 'bookid_88'."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps purchase IDs to book IDs using a known rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {purchase_id}: {e}")
        return None

predicted_book_ids = []
for i, row in df_review.iterrows():
    purchase_id = row["purchase_id"]
    book_id = resolve_purchase_to_book(purchase_id)
    predicted_book_ids.append(book_id)
    print(f"[{i}] purchase_id = {purchase_id} → book_id = {book_id}")

df_review["book_id"] = predicted_book_ids


✅ Loaded 1833 reviews

🧠 GPT-inferred mapping rule:

1. **Mapping Rule**: The mapping between `purchase_id` and `book_id` is based on the numerical suffix of each ID. Specifically, the `purchase_id` with the suffix `N` corresponds to the `book_id` with the same numerical suffix `N`. For example, `purchaseid_1` maps to `bookid_1`, `purchaseid_2` maps to `bookid_2`, and so on, up to `purchaseid_200` mapping to `bookid_200`.

2. **Explanation**: This rule holds because both `purchase_id` and `book_id` follow a consistent pattern where the prefix (i.e., "purchaseid_" and "bookid_") is constant, and only the numerical part varies. Since both lists contain IDs numbered from 1 to 200, it is logical to conclude that each `purchase_id` directly corresponds to a `book_id` with the same numerical identifier. The absence of any gaps or discrepancies in the numbering further supports this mapping rule.
[0] purchase_id = purchaseid_186 → book_id = bookid_186
[1] purchase_id = purchaseid_191 → book_i

In [ ]:
import pandas as pd


# ===== Step 3: 清洗评分列 =====
df_review["rating"] = pd.to_numeric(df_review["rating"], errors="coerce")
df_review = df_review[df_review["rating"].notnull()]

# ===== Step 4: 只保留出现在 df_qualified 中的书 =====
qualified_book_ids = set(df_qualified["book_id"])
df_filtered = df_review[df_review["book_id"].isin(qualified_book_ids)].copy()

# ===== Step 5: 按 book_id 计算平均评分 =====
df_grouped = (
    df_filtered.groupby("book_id")
    .agg(avg_rating=("rating", "mean"), num_reviews=("rating", "count"))
    .reset_index()
)

# ===== Step 6: 只保留平均评分恰好为 5.0 的书 =====
df_perfect = df_grouped[df_grouped["avg_rating"] == 5.0].copy()

# ===== Step 7: 加入书名（title）信息 =====
# Step 7: 加入 title 和 categories 信息
df_result = df_perfect.merge(
    df_qualified[["book_id", "title", "categories"]], on="book_id", how="left"
)

# Step 7.5: 进一步过滤，确保 categories 明确包含 'Literature & Fiction'
df_result = df_result[df_result["categories"].str.contains("Literature & Fiction", case=False, na=False)]

# Step 8: 输出结果
print("📚 English Literature & Fiction books with perfect average rating of 5.0 (verified categories):")
print(df_result[["title", "categories"]])


📚 English Literature & Fiction books with perfect average rating of 5.0 (verified categories):
                                                title  \
0            Knowing When To Die: Uncollected Stories   
1                              Childe Harold of Dysna   
2                          Forged in Blood (Freehold)   
3                        Exits, Desires, & Slow Fires   
4                                   Kennebago Moments   
5                                          The Sludge   
6                                     Liza of Lambeth   
7   Something That Feels Like Truth (Switchgrass B...   
8   The Prophet: With Original 1923 Illustrations ...   
9                      The Melancholy Strumpet Master   
10  Child Of The King A Journey of Hope Book 1: Ea...   
11                                       Fire Cracker   
12                                        Local Honey   
13           Reunion: The Children of Lauderdale Park   
14  Hollywood Confessions: Hollywood Headlines Boo